In [ ]:
"""
Vinicius Costa Gandolfi
"""

!wget -N -q "https://ampl.com/dl/open/couenne/couenne-linux64.zip" #MINLP grande
!unzip -o -q couenne-linux64
!pip install -q pyomo
import numpy as np
import pandas as pd
from pyomo.environ import *

     |████████████████████████████████| 9.1 MB 4.2 MB/s 
     |████████████████████████████████| 49 kB 4.9 MB/s 


In [ ]:
DM_df = pd.DataFrame({1:[0, 22, 53, 53], 2:[22, 0, 40, 62], 3:[53, 40, 0, 55], 4:[53, 62, 55, 0]},
                  columns=np.arange(1, 5), index=np.arange(1, 5)) # Distancias entre as máquinas
DM_df

,1,2,3,4
1,0,22,53,53
2,22,0,40,62
3,53,40,0,55
4,53,62,55,0


In [ ]:
Fl_df = pd.DataFrame({1:[0, 3, 0, 2], 2:[3, 0, 0, 1], 3:[0, 0, 0, 4], 4:[2, 1, 4, 0]},
                  columns=np.arange(1, 5), index=np.arange(1, 5))# Facilidades
Fl_df

,1,2,3,4
1,0,3,0,2
2,3,0,0,1
3,0,0,0,4
4,2,1,4,0


In [ ]:
DM = DM_df.stack().to_dict()
DM

{(1, 1): 0,
 (1, 2): 22,
 (1, 3): 53,
 (1, 4): 53,
 (2, 1): 22,
 (2, 2): 0,
 (2, 3): 40,
 (2, 4): 62,
 (3, 1): 53,
 (3, 2): 40,
 (3, 3): 0,
 (3, 4): 55,
 (4, 1): 53,
 (4, 2): 62,
 (4, 3): 55,
 (4, 4): 0}

In [ ]:
Fl = Fl_df.stack().to_dict()
Fl

{(1, 1): 0,
 (1, 2): 3,
 (1, 3): 0,
 (1, 4): 2,
 (2, 1): 3,
 (2, 2): 0,
 (2, 3): 0,
 (2, 4): 1,
 (3, 1): 0,
 (3, 2): 0,
 (3, 3): 0,
 (3, 4): 4,
 (4, 1): 2,
 (4, 2): 1,
 (4, 3): 4,
 (4, 4): 0}

In [ ]:
I = np.arange(1, 5)
I

array([1, 2, 3, 4])

In [ ]:
J = np.arange(1, 5)
J

array([1, 2, 3, 4])

# Quadract Assignmant Problemn

$\sum_{i=0}^n$ $\sum_{j=0}^n$ $f_{ij}$ * $x_{ij}$ *$ d_{ij}$ * $x_{ij}$ 

$\sum_{i=1}^n x_{ij} = 1 ∀ i = 1... n$ 

$\sum_{j=1}^n x_{ij} = 1 ∀ i = 1... n$ 

$x_i$ = {0, 1} ∀ j = 1... n

In [ ]:
# Inicializa o problema p que será resolvido com o solver 
model = ConcreteModel()

In [ ]:
# Maximização
# Fazendo os Sets
model.I = Set(initialize=I)
model.J = Set(initialize=J)

In [ ]:
model.DM = Param(model.I, model.J, initialize=DM)
model.Fl = Param(model.I, model.J, initialize=Fl)
model.b = Param(initialize=1)

In [ ]:
model.x = Var(model.I, model.J, domain=Binary, bounds=(0, 1))

In [ ]:
expr = np.arange(0, 10)
expr = sum(expr)
expr

45

In [ ]:
expr = sum(model.DM[i, j]  * model.Fl[i, j] * model.x[i, j] * model.x[i, j] for i in model.I for j in model.J if i!= j)
model.obj = Objective(sense=minimize, expr=expr)

In [ ]:
model.rest1 = ConstraintList()
for j in model.J:
  a = sum(model.x[i, j] for i in model.I if i!=j)
  b = model.b
  model.rest1.add(a == b)

In [ ]:
model.rest2 = ConstraintList()
for i in model.I:
  a = sum(model.x[i, j] for j in model.J if i!=j)
  b = model.b
  model.rest2.add(a == b)

In [ ]:
model.pprint()

7 Set Declarations
    DM_index : Size=1, Index=None, Ordered=True
        Key  : Dimen : Domain : Size : Members
        None :     2 :    I*J :   16 : {(1, 1), (1, 2), (1, 3), (1, 4), (2, 1), (2, 2), (2, 3), (2, 4), (3, 1), (3, 2), (3, 3), (3, 4), (4, 1), (4, 2), (4, 3), (4, 4)}
    Fl_index : Size=1, Index=None, Ordered=True
        Key  : Dimen : Domain : Size : Members
        None :     2 :    I*J :   16 : {(1, 1), (1, 2), (1, 3), (1, 4), (2, 1), (2, 2), (2, 3), (2, 4), (3, 1), (3, 2), (3, 3), (3, 4), (4, 1), (4, 2), (4, 3), (4, 4)}
    I : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {1, 2, 3, 4}
    J : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {1, 2, 3, 4}
    rest1_index : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {1, 2, 3, 4}
    rest2_

In [ ]:
solver = SolverFactory('couenne', executable='/content/couenne')
results = solver.solve(model, tee=True)

Couenne 0.5.7 -- an Open-Source solver for Mixed Integer Nonlinear Optimization
Mailing list: couenne@list.coin-or.org
Instructions: http://www.coin-or.org/Couenne
couenne: 
ANALYSIS TEST: NLP0012I 
              Num      Status      Obj             It       time                 Location
NLP0014I             1         OPT 68.824313        5 0.001898
NLP0014I             2      INFEAS 190        0 0
Loaded instance "/tmp/tmp8o_pqwxu.pyomo.nl"
Constraints:            8
Variables:             12 (12 integer)
Auxiliaries:           10 (10 integer)

Coin0506I Presolve 32 (-1) rows, 20 (-2) columns and 72 (-9) elements
Clp0006I 0  Obj 0 Primal inf 7.999992 (8)
Clp0006I 14  Obj 124
Clp0000I Optimal - objective value 124
Clp0032I Optimal objective 124 - 14 iterations time 0.002, Presolve 0.00
Clp0000I Optimal - objective value 124
Couenne: new cutoff value 1.2400000000e+02 (0.008037 seconds)
NLP Heuristic: time limit reached.
Clp0000I Optimal - objective value 124
Optimality Based BT: 0 improv

In [ ]:
#df = pd.DataFrame(index=pd.MultiIndex.from_tuples(model.x, names=['i', 'j']))
#df['x'] = [value(model.x[key]) for key in df.index]

df = pd.DataFrame(index=I, columns=J)
for i in model.I:
  for j in model.J:
    if i != j:
      df[i][j] = value(model.x[i, j])
    else:
      df[i][j] = "0"
df = df.add_prefix("x")
df

,x1,x2,x3,x4
1,0,0,1,0
2,0,0,0,1
3,1,0,0,0
4,0,1,0,0


In [ ]:
print(f"Valor Mínimo da função: {model.obj()}")

Valor Mínimo da função: 124.0
